In [10]:
"""
Two-Stage Classifier Training and Evaluation

This notebook trains and evaluates a two-stage classifier for ECoG signal classification:
- Stage 1: Binary classifier (silence vs stimulus)
- Stage 2: Frequency classifier (8 tone frequencies)

Designed for low-latency, causal inference with a 50ms history buffer.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import seaborn as sns

# Project imports
import sys
sys.path.insert(0, str(Path.cwd().parent))

from brainstorm.loading import load_raw_data, load_channel_coordinates
from brainstorm.evaluation import ModelEvaluator
from brainstorm.ml.two_stage_classifier import TwoStageClassifier

plt.style.use('fivethirtyeight')
%matplotlib inline

In [12]:
# Load data
DATA_PATH = Path("/Users/raghavg/Desktop/mind_meld/data")

train_features, train_labels = load_raw_data(DATA_PATH, step="train")
val_features, val_labels = load_raw_data(DATA_PATH, step="validation")

print(f"Training: {train_features.shape[0]} samples")
print(f"Validation: {val_features.shape[0]} samples")
print(f"\nLabel distribution (training):")
print(train_labels['label'].value_counts().sort_index())

2026-01-23 14:04:27.543 | INFO     | brainstorm.loading:load_raw_data:38 - Loading data from: /Users/raghavg/Desktop/mind_meld/data
2026-01-23 14:04:28.599 | INFO     | brainstorm.loading:load_raw_data:50 - Features shape: (90386, 1024)
2026-01-23 14:04:28.609 | INFO     | brainstorm.loading:load_raw_data:51 - Labels shape: (90386, 1)
2026-01-23 14:04:28.612 | INFO     | brainstorm.loading:load_raw_data:54 - Features time range: 0.0 to 90.385
2026-01-23 14:04:28.617 | INFO     | brainstorm.loading:load_raw_data:57 - Labels time range: 0.0 to 90.385
2026-01-23 14:04:28.624 | INFO     | brainstorm.loading:load_raw_data:38 - Loading data from: /Users/raghavg/Desktop/mind_meld/data
2026-01-23 14:04:28.820 | INFO     | brainstorm.loading:load_raw_data:50 - Features shape: (22596, 1024)
2026-01-23 14:04:28.821 | INFO     | brainstorm.loading:load_raw_data:51 - Labels shape: (22596, 1)
2026-01-23 14:04:28.821 | INFO     | brainstorm.loading:load_raw_data:54 - Features time range: 90.387 to 11

Training: 90386 samples
Validation: 22596 samples

Label distribution (training):
label
0.0       60454
120.0      3670
224.0      3668
421.0      3674
789.0      3853
1479.0     3851
2772.0     3853
5195.0     3697
9736.0     3666
Name: count, dtype: int64


In [13]:
# Initialize and train the two-stage classifier
model = TwoStageClassifier(
    history_size=100,      # 50ms history buffer
    hidden_size=64,      # Small hidden layer for compact model
    dropout=0.3,
)

# Train the model
model.fit(
    X=train_features.values,
    y=train_labels["label"].values,
    epochs=50,
    batch_size=256,
    learning_rate=1e-3,
)

2026-01-23 14:05:22.527 | INFO     | brainstorm.ml.base:fit:70 - Training TwoStageClassifier...
2026-01-23 14:05:22.620 | INFO     | brainstorm.ml.two_stage_classifier:fit_model:259 - Stage 1: Binary classification (silence vs stimulus)
2026-01-23 14:05:22.621 | INFO     | brainstorm.ml.two_stage_classifier:fit_model:260 - Stage 2: 8 frequency classes: [120.0, 224.0, 421.0, 789.0, 1479.0, 2772.0, 5195.0, 9736.0]
2026-01-23 14:05:23.185 | INFO     | brainstorm.ml.two_stage_classifier:fit_model:274 - Extracting history features...
2026-01-23 14:05:31.325 | INFO     | brainstorm.ml.two_stage_classifier:fit_model:281 - Training Stage 1 (silence vs stimulus)...
Stage 1: 100%|██████████| 50/50 [00:37<00:00,  1.34it/s, loss=0.0467]
2026-01-23 14:06:11.765 | INFO     | brainstorm.ml.two_stage_classifier:_train_stage:373 - Stage 1 training complete. Final loss: 0.0467
2026-01-23 14:06:11.766 | INFO     | brainstorm.ml.two_stage_classifier:fit_model:294 - Training Stage 2 (frequency classificati

In [18]:
# Check model size
import os
model_path = Path("/Users/raghavg/Desktop/mind_meld/models")
model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model size: {model_size_mb:.2f} MB")

Model size: 0.00 MB


In [15]:
# Evaluate using the official ModelEvaluator
evaluator = ModelEvaluator(
    test_features=val_features,
    test_labels=val_labels[["label"]],
)
results = evaluator.evaluate()

2026-01-23 14:06:39.515 | INFO     | brainstorm.evaluation:_load_model:84 - Loading model from: model.pt
2026-01-23 14:06:39.516 | INFO     | brainstorm.evaluation:_load_model:85 - Using class: brainstorm.ml.two_stage_classifier.TwoStageClassifier


FileNotFoundError: Model file not found: model.pt

In [ ]:
# Manual evaluation to get detailed metrics
# Reset buffer before evaluation
model_loaded = TwoStageClassifier.load()
model_loaded.reset_buffer()

# Get predictions
y_true = val_labels["label"].values
y_pred = []

for i in range(len(val_features)):
    sample = val_features.iloc[i].values
    pred = model_loaded.predict(sample)
    y_pred.append(pred)

y_pred = np.array(y_pred)

# Calculate balanced accuracy
balanced_acc = balanced_accuracy_score(y_true, y_pred)
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Balanced Accuracy Score (out of 50): {balanced_acc * 50:.2f}")

In [ ]:
# Classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
# Confusion matrix visualization
unique_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
label_names = [str(int(l)) if l > 0 else "silence" for l in unique_labels]

cm = confusion_matrix(y_true, y_pred, labels=unique_labels)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (counts)')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (normalized)')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy analysis
print("Per-Class Accuracy:")
print("=" * 40)
for label in unique_labels:
    mask = y_true == label
    class_acc = (y_pred[mask] == label).mean()
    label_name = "silence" if label == 0 else f"{int(label)} Hz"
    print(f"  {label_name:12s}: {class_acc:.4f}")

In [ ]:
# Stage 1 analysis: How well does it detect stimulus vs silence?
binary_true = (y_true != 0).astype(int)
binary_pred = (y_pred != 0).astype(int)

stage1_acc = (binary_pred == binary_true).mean()
stage1_precision = binary_pred[binary_true == 1].mean() if binary_true.sum() > 0 else 0
stage1_recall = binary_true[binary_pred == 1].mean() if binary_pred.sum() > 0 else 0

print("Stage 1 Performance (Stimulus Detection):")
print("=" * 40)
print(f"  Accuracy: {stage1_acc:.4f}")
print(f"  Precision (when predicting stimulus): {stage1_precision:.4f}")
print(f"  Recall (detecting true stimulus): {stage1_recall:.4f}")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("BASELINE TWO-STAGE CLASSIFIER SUMMARY")
print("=" * 60)
print(f"\nModel Configuration:")
print(f"  History buffer: 50 samples (50ms)")
print(f"  Hidden size: 128")
print(f"  Model size: {model_size_mb:.2f} MB")

print(f"\nMetrics:")
print(f"  Balanced Accuracy: {balanced_acc:.4f}")
print(f"  Stage 1 Accuracy (stimulus detection): {stage1_acc:.4f}")

print(f"\nEstimated Score Components:")
print(f"  Accuracy Score (50%): {balanced_acc * 50:.2f} / 50")
print(f"  Size Score: ~{np.exp(-4 * model_size_mb / 5) * 25:.2f} / 25 (based on {model_size_mb:.2f} MB)")
print(f"  Lag Score: TBD (depends on evaluation)")
print("=" * 60)